# Bài 23: Khớp mô hình, Ước lượng bền vững và RANSAC (Model Fitting, Robust Estimation, and RANSAC)

Nhiều bài toán thị giác máy tính quy về việc khớp một mô hình &mdash; một đường thẳng, một đường tròn, một mặt phẳng, một phép đồng nhất (homography) (Bài 24), một ma trận cơ bản (fundamental matrix) (Bài 28) &mdash; với một tập các điểm tương ứng. Trên thực tế, dữ liệu đó gần như luôn chứa các điểm ngoại lai: các cặp tương ứng đơn giản là sai. Bài này đối mặt trực tiếp với điều đó: phép khớp bình phương tối thiểu mong manh một cách thảm hại chỉ với vài điểm xấu, và **RANSAC** là cách khắc phục tiêu chuẩn.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Bình phương tối thiểu thông thường và toàn phần

Khớp $y=mx+b$ bằng cách cực tiểu hóa $\sum(y_i-(mx_i+b))^2$ &mdash; **bình phương tối thiểu thông thường (ordinary least squares — OLS)** &mdash; đo sai số theo phương *thẳng đứng*. Đó là lựa chọn đúng khi $x$ được biết chính xác và chỉ $y$ có nhiễu, nhưng không đúng khi cả hai tọa độ mang nhiễu tương đương, như thường thấy với các điểm ảnh 2D: phép khớp bị lệch về phía hướng nào có độ trải ít hơn. (Và nó thậm chí không biểu diễn được đường thẳng đứng.) **Bình phương tối thiểu toàn phần (total least squares — TLS)** thay vào đó cực tiểu hóa khoảng cách *vuông góc* từ mỗi điểm tới đường thẳng, đối xử với cả hai tọa độ một cách đối xứng.

Với đường thẳng 2D, TLS không cần công cụ mới: đường khớp tốt nhất đi qua trọng tâm, và hướng của nó là vector riêng ứng với giá trị riêng *lớn nhất* của ma trận hiệp phương sai các điểm (đã trừ tâm) &mdash; chính là ý tưởng hiệp-phương-sai-và-vector-riêng ở Bài 6, chỉ là chọn ra trục có độ trải lớn nhất thay vì mô tả hình dạng của một đốm. (Với nhiều ẩn số hơn &mdash; như với phép đồng nhất hay ma trận cơ bản &mdash; cách tiếp cận này tổng quát hóa thành **phân tích giá trị suy biến (singular value decomposition — SVD)**, được giới thiệu ở Bài 28.)

In [ ]:
rng_tls = np.random.default_rng(2)
true_m, true_b = 3.0, 1.0  # a steep line, where OLS struggles most
t = rng_tls.uniform(0, 5, 40)
x_noisy = t + rng_tls.normal(0, 0.4, 40)
y_noisy = true_m * t + true_b + rng_tls.normal(0, 0.4, 40)

A_line = np.vstack([x_noisy, np.ones_like(x_noisy)]).T
slope_ols, intercept_ols = np.linalg.lstsq(A_line, y_noisy, rcond=None)[0]

points = np.stack([x_noisy, y_noisy], axis=1)
centroid = points.mean(axis=0)
cov = np.cov((points - centroid).T)
eigvals, eigvecs = np.linalg.eigh(cov)          # ascending order
direction = eigvecs[:, -1]                      # largest eigenvalue = direction of max spread
slope_tls = direction[1] / direction[0]
intercept_tls = centroid[1] - slope_tls * centroid[0]

print(f'true line:  y = {true_m:.2f}x + {true_b:.2f}')
print(f'OLS fit:    y = {slope_ols:.2f}x + {intercept_ols:.2f}   <- biased, since x is noisy too')
print(f'TLS fit:    y = {slope_tls:.2f}x + {intercept_tls:.2f}   <- closer to the true slope')

xs_tls = np.array([x_noisy.min(), x_noisy.max()])
plt.scatter(x_noisy, y_noisy, s=15, alpha=0.6, label='noisy points (both x and y)')
plt.plot(xs_tls, true_m * xs_tls + true_b, '--', color='gray', label='true line')
plt.plot(xs_tls, slope_ols * xs_tls + intercept_ols, color='tab:orange', label='OLS (vertical error)')
plt.plot(xs_tls, slope_tls * xs_tls + intercept_tls, color='tab:green', label='TLS (perpendicular error)')
plt.legend(fontsize=8)
plt.title('OLS vs. TLS when both x and y are noisy')
plt.show()

Ở đây TLS khôi phục độ dốc thực tốt hơn thấy rõ, vì nhiễu thực sự đối xứng theo $x$ và $y$.

## Tại sao bình phương tối thiểu thất bại khi có ngoại lai

Phép khớp bình phương tối thiểu cực tiểu hóa *tổng bình phương* các phần dư. Nhưng việc bình phương có một nhược điểm: mỗi điểm ngoại lai (điểm nằm xa mô hình) đóng góp một lượng khổng lồ vào tổng sai số, khiến toàn bộ phép khớp bị kéo ra xa khỏi tất cả các điểm khác, vốn đúng.

In [ ]:
rng = np.random.default_rng(0)
true_slope, true_intercept = 2.0, 5.0

x_inliers = rng.uniform(0, 10, 40)
y_inliers = true_slope * x_inliers + true_intercept + rng.normal(0, 0.5, 40)

x_outliers = rng.uniform(0, 10, 15)
y_outliers = rng.uniform(-20, 40, 15)  # unrelated to the line at all

x = np.concatenate([x_inliers, x_outliers])
y = np.concatenate([y_inliers, y_outliers])

A = np.vstack([x, np.ones_like(x)]).T
slope_ols, intercept_ols = np.linalg.lstsq(A, y, rcond=None)[0]

print(f'true line:                 y = {true_slope:.2f}x + {true_intercept:.2f}')
print(f'ordinary least squares:    y = {slope_ols:.2f}x + {intercept_ols:.2f}   <- dragged off by outliers')

xs = np.array([0, 10])
plt.scatter(x_inliers, y_inliers, c='tab:blue', label='inliers')
plt.scatter(x_outliers, y_outliers, c='tab:red', marker='x', label='outliers')
plt.plot(xs, true_slope * xs + true_intercept, '--', color='gray', label='true line')
plt.plot(xs, slope_ols * xs + intercept_ols, color='tab:orange', label='OLS fit')
plt.legend(fontsize=8)
plt.title('Ordinary least squares, corrupted by 15 outliers among 40 inliers')
plt.show()

## RANSAC: khớp từ trong ra ngoài

**RANSAC** (RANdom SAmple Consensus — đồng thuận mẫu ngẫu nhiên, <a href="../references.html#fischler-bolles-1981">Fischler & Bolles, 1981</a><span class="landmark-paper">&#9733;</span>) đảo ngược chiến lược: thay vì dùng *toàn bộ* dữ liệu và hy vọng ngoại lai không quan trọng, nó lặp đi lặp lại việc chọn tập con ngẫu nhiên *nhỏ nhất có thể* đủ để xác định một mô hình ứng viên, đếm xem bao nhiêu điểm còn lại đồng ý với nó (gọi là **tập đồng thuận (consensus set)**), và giữ lại ứng viên nào có nhiều sự đồng ý nhất. Một lần khớp lại bình phương tối thiểu cuối cùng trên tập điểm trong (inlier) chiến thắng cho ra kết quả được trau chuốt (vì bình phương tối thiểu hoạt động tốt khi không có ngoại lai).

In [ ]:
def ransac_line(x, y, threshold=2.0, n_iterations=200, rng=None):
    rng = rng or np.random.default_rng()
    best_inliers, best_count = None, -1

    for _ in range(n_iterations):
        i, j = rng.choice(len(x), 2, replace=False)     # minimal sample: 2 points define a line
        if x[i] == x[j]:
            continue
        m = (y[j] - y[i]) / (x[j] - x[i])
        b = y[i] - m * x[i]
        distance = np.abs(m * x - y + b) / np.sqrt(m**2 + 1)
        inliers = distance < threshold
        if inliers.sum() > best_count:
            best_count, best_inliers = inliers.sum(), inliers

    A = np.vstack([x[best_inliers], np.ones(best_inliers.sum())]).T
    slope, intercept = np.linalg.lstsq(A, y[best_inliers], rcond=None)[0]  # final refit, inliers only
    return slope, intercept, best_inliers

slope_ransac, intercept_ransac, inlier_mask = ransac_line(x, y, rng=np.random.default_rng(1))

print(f'true line:   y = {true_slope:.2f}x + {true_intercept:.2f}')
print(f'RANSAC fit:  y = {slope_ransac:.2f}x + {intercept_ransac:.2f}')
print(f'inliers found: {inlier_mask.sum()} / {len(x)}  (planted {40} true inliers)')

plt.scatter(x[inlier_mask], y[inlier_mask], c='tab:blue', label='found inliers')
plt.scatter(x[~inlier_mask], y[~inlier_mask], c='tab:red', marker='x', label='rejected as outliers')
plt.plot(xs, slope_ransac * xs + intercept_ransac, color='tab:green', label='RANSAC fit')
plt.legend(fontsize=8)
plt.title('RANSAC recovers the true line despite 27% outlier contamination')
plt.show()

Ảnh động này chạy `ransac_line` trên chính dữ liệu đó: mỗi lần thử lấy mẫu 2 điểm ngẫu nhiên (các vòng tròn cam), khớp một đường thẳng qua chúng, và tô màu mọi điểm theo việc nó có nằm trong phạm vi `threshold` của ứng viên đó hay không. Hầu hết các lần thử rơi vào một đường thẳng tồi với ít điểm trong; đường nét đứt xanh lục chỉ cập nhật khi một ứng viên vượt qua ứng viên tốt nhất hiện tại, và khung hình cuối cho thấy lần khớp lại bình phương tối thiểu trên tập điểm trong chiến thắng.

<img src="../animations/ransac_demo.gif" alt="Ảnh động RANSAC: lặp lại việc lấy mẫu 2 điểm ngẫu nhiên, khớp một đường thẳng ứng viên, đếm số điểm trong, và giữ mô hình tốt nhất cho đến lần khớp lại bình phương tối thiểu cuối cùng trên tập điểm trong chiến thắng" style="display: block; margin: 0 auto; max-width: 680px;">

## RANSAC cần bao nhiêu vòng lặp?

Nếu một tỉ lệ $w$ của dữ liệu là điểm trong, và mô hình cần một mẫu tối thiểu gồm $n$ điểm, xác suất để một mẫu ngẫu nhiên bất kỳ toàn là điểm trong là $w^n$. Để tin chắc ít nhất `p` rằng ta rút được một mẫu toàn-điểm-trong ít nhất một lần qua $N$ lần thử độc lập:

$$N = \frac{\log(1-p)}{\log(1-w^n)}$$

Càng nhiều ngoại lai, hoặc kích thước mẫu tối thiểu càng lớn, thì số mẫu cần thiết này càng tăng.

In [ ]:
def required_iterations(inlier_fraction, sample_size, confidence=0.99):
    return np.log(1 - confidence) / np.log(1 - inlier_fraction**sample_size)

print(f'{"inlier %":>10} {"line (n=2)":>12} {"homography (n=4)":>18} {"fundamental matrix (n=8)":>18}')
for w in [0.9, 0.7, 0.5, 0.3]:
    n_line = int(np.ceil(required_iterations(w, 2)))
    n_homog = int(np.ceil(required_iterations(w, 4)))
    n_fund = int(np.ceil(required_iterations(w, 8)))
    print(f'{100*w:>9.0f}% {n_line:>12} {n_homog:>18} {n_fund:>18}')

Khớp một đường thẳng chỉ cần 2 điểm, nên ngay cả mức nhiễm bẩn khá nặng (50% ngoại lai) cũng chỉ cần vài chục vòng lặp. Khớp một phép đồng nhất cần mẫu tối thiểu 4 điểm, nên cùng tỉ lệ điểm trong đó cần số vòng lặp lớn hơn cả một bậc độ lớn &mdash; cái giá của một mô hình phức tạp hơn. Tương tự với việc khớp ma trận cơ bản, vốn cần mẫu tối thiểu 8 điểm.

## Một phương án nhẹ nhàng hơn: các hàm mất mát bền vững

RANSAC đưa ra một quyết định cứng điểm trong/ngoại lai. Một họ phương pháp thay thế, **M-estimator**, thay vào đó đánh trọng số lại đóng góp của mọi điểm một cách mượt mà &mdash; ví dụ **hàm mất mát Huber (Huber loss)** hoạt động như sai số bình phương thông thường với phần dư nhỏ nhưng chuyển sang tăng tuyến tính (ít mạnh tay hơn nhiều) khi vượt quá một ngưỡng, nên một điểm sai lệch rất nhiều không còn có thể chi phối tổng chi phí như trong `Loss = residual^2`. Trên thực tế RANSAC và M-estimator bổ sung cho nhau: RANSAC rất giỏi loại bỏ các ngoại lai *thô* (các cặp khớp hoàn toàn sai), trong khi một bước tinh chỉnh M-estimator sau đó có thể giảm trọng số các sai lệch nhỏ hơn, tinh tế hơn giữa các điểm trong còn lại.

In [ ]:
def huber_reweighted_fit(x, y, delta=2.0, n_iterations=10):
    weights = np.ones_like(x)
    for _ in range(n_iterations):
        sw = np.sqrt(weights)
        A = np.vstack([x, np.ones_like(x)]).T * sw[:, None]  # weighted least squares
        slope, intercept = np.linalg.lstsq(A, y * sw, rcond=None)[0]
        residuals = np.abs(y - (slope * x + intercept))
        weights = np.where(residuals > delta, delta / np.maximum(residuals, 1e-6), 1.0)
    return slope, intercept

slope_huber, intercept_huber = huber_reweighted_fit(x, y)

print(f'true line:   y = {true_slope:.2f}x + {true_intercept:.2f}')
print(f'OLS fit:     y = {slope_ols:.2f}x + {intercept_ols:.2f}')
print(f'Huber fit:   y = {slope_huber:.2f}x + {intercept_huber:.2f}')
print(f'RANSAC fit:  y = {slope_ransac:.2f}x + {intercept_ransac:.2f}')

plt.scatter(x_inliers, y_inliers, c='tab:blue', label='inliers')
plt.scatter(x_outliers, y_outliers, c='tab:red', marker='x', label='outliers')
plt.plot(xs, true_slope * xs + true_intercept, '--', color='gray', label='true line')
plt.plot(xs, slope_ols * xs + intercept_ols, color='tab:orange', label='OLS')
plt.plot(xs, slope_huber * xs + intercept_huber, color='tab:purple', label='Huber-reweighted')
plt.plot(xs, slope_ransac * xs + intercept_ransac, color='tab:green', label='RANSAC')
plt.legend(fontsize=8)
plt.title('Same contaminated data, three different fits')
plt.show()

Phép khớp Huber nằm giữa hai cách kia: gần đường thẳng thực hơn nhiều so với OLS, dù không hoàn toàn chính xác bằng RANSAC trên dữ liệu bị nhiễm bẩn nặng như thế này, vì một nhúm ngoại lai thô vẫn kéo nhẹ ở mỗi vòng lặp thay vì bị cắt bỏ hoàn toàn.

### Bài tập

1. Tăng số ngoại lai của `ransac_line` cho đến khi khoảng 70% số điểm là ngoại lai. 200 vòng lặp có còn đủ để khôi phục đường thẳng thực một cách đáng tin cậy không? Dùng công thức `required_iterations` để kiểm tra xem về lý thuyết 200 có đủ ở mức nhiễm bẩn đó không.
2. Hạ `threshold` khoảng cách của `ransac_line` từ `2.0` xuống `0.5`. Số điểm trong tìm được có thay đổi không, và tại sao một ngưỡng quá chặt lại có thể bắt đầu loại bỏ cả các điểm trong *thật sự* (gợi ý: hãy nghĩ xem nhiễu, chứ không phải ngoại lai, làm gì với các điểm đúng)?
3. Thử `huber_reweighted_fit` với `delta` nhỏ hơn nhiều (ví dụ `0.5`) và lớn hơn nhiều (ví dụ `5.0`). `delta` đánh đổi thế nào giữa hành vi giống OLS (hầu như không giảm trọng số) và loại bỏ cứng giống RANSAC (gần như nhị phân trong/ngoài)?